Read the csv file and change it to DataFrame

In [1]:
import pandas as pd

data = pd.read_csv('dirty_cafe_sales.csv')

df = pd.DataFrame(data)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


First Changing the Columns names

In [2]:
df = df.rename(columns={
    'Transaction ID': 'Transaction_ID',
    'Price Per Unit': 'Price_Per_Unit',
    'Total Spent': 'Total_Spent',
    'Payment Method': 'Payment_Method',
    'Transaction Date': 'Transaction_Date'
})

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price_Per_Unit    9821 non-null   str  
 4   Total_Spent       9827 non-null   str  
 5   Payment_Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction_Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


Drop Duplicates

In [3]:
df.drop_duplicates(inplace=True)

Then see where are the null values and fill them with correct values

In [4]:
df['Location'] = df['Location'].replace('In-store', 'In_Store')
df['Location'] = df['Location'].replace('UNKNOWN', 'In_Store')
df['Location'] = df['Location'].fillna('In_Store')

df['Payment_Method'] = df['Payment_Method'].replace('UNKNOWN', 'Cash')
df['Payment_Method'] = df['Payment_Method'].fillna('Cash')

df.isna().sum()

Transaction_ID        0
Item                333
Quantity            138
Price_Per_Unit      179
Total_Spent         173
Payment_Method        0
Location              0
Transaction_Date    159
dtype: int64

After fixing the null values of payment method and location i have to know if there is another wrong value in them

In [5]:
df['Location'].value_counts()

Location
In_Store    6620
Takeaway    3022
ERROR        358
Name: count, dtype: int64

In [6]:
df['Location'] = df['Location'].replace('ERROR', 'Takeaway')

df['Location'].value_counts()

Location
In_Store    6620
Takeaway    3380
Name: count, dtype: int64

Know the payment

In [7]:
df['Payment_Method'].value_counts()

Payment_Method
Cash              5130
Digital Wallet    2291
Credit Card       2273
ERROR              306
Name: count, dtype: int64

In [8]:
df['Payment_Method'] = df['Payment_Method'].replace('ERROR', 'Cash')
df['Payment_Method'] = df['Payment_Method'].replace('Digital Wallet', 'Digital_Wallet')
df['Payment_Method'] = df['Payment_Method'].replace('Credit Card', 'Credit_Card')

df['Payment_Method'].value_counts()

Payment_Method
Cash              5436
Digital_Wallet    2291
Credit_Card       2273
Name: count, dtype: int64

I have fixed the whole payment method and the location know i will clean the Date

In [9]:
#I will fix the type of all Series
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price_Per_Unit'] = pd.to_numeric(df['Price_Per_Unit'], errors='coerce')
df['Total_Spent'] = pd.to_numeric(df['Total_Spent'], errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    10000 non-null  str           
 1   Item              9667 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price_Per_Unit    9467 non-null   float64       
 4   Total_Spent       9498 non-null   float64       
 5   Payment_Method    10000 non-null  str           
 6   Location          10000 non-null  str           
 7   Transaction_Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [10]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'wrong_Date'

df['Season'] = df['Transaction_Date'].dt.month.apply(get_season)

df.head()

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date,Season
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit_Card,Takeaway,2023-09-08,Fall
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In_Store,2023-05-16,Spring
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit_Card,In_Store,2023-07-19,Summer
3,TXN_7034554,Salad,2.0,5.0,10.0,Cash,In_Store,2023-04-27,Spring
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital_Wallet,In_Store,2023-06-11,Summer


In [11]:
df = df.dropna(subset=['Transaction_Date'])
df

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date,Season
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit_Card,Takeaway,2023-09-08,Fall
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In_Store,2023-05-16,Spring
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit_Card,In_Store,2023-07-19,Summer
3,TXN_7034554,Salad,2.0,5.0,10.0,Cash,In_Store,2023-04-27,Spring
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital_Wallet,In_Store,2023-06-11,Summer
...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Cash,In_Store,2023-08-30,Summer
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital_Wallet,In_Store,2023-06-02,Summer
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital_Wallet,In_Store,2023-03-02,Spring
9998,TXN_7695629,Cookie,3.0,NaN,3.0,Digital_Wallet,In_Store,2023-12-02,Winter


Know cleaning the Quantity, the Price per unit, and total spent

In [12]:
df.groupby(['Item', 'Price_Per_Unit'])['Transaction_ID'].count()

Item      Price_Per_Unit
Cake      3.0               1031
Coffee    2.0               1069
Cookie    1.0                976
ERROR     1.0                 33
          1.5                 37
          2.0                 31
          3.0                 75
          4.0                 58
          5.0                 38
Juice     3.0               1065
Salad     5.0               1035
Sandwich  4.0               1027
Smoothie  4.0                990
Tea       1.5                965
UNKNOWN   1.0                 43
          1.5                 39
          2.0                 47
          3.0                 75
          4.0                 67
          5.0                 41
Name: Transaction_ID, dtype: int64

In [13]:
df['Item'].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

Making the item based on the price

In [14]:
def right_item(row):
    if pd.isna(row['Item']) or row['Item'] in ['ERROR', 'UNKNOWN']:
        if row['Price_Per_Unit'] == 3.0:
            return 'Cake'
        elif row['Price_Per_Unit'] == 2.0:
            return 'Coffee'
        elif row['Price_Per_Unit'] == 5.0:
            return 'Salad'
        elif row['Price_Per_Unit'] == 1.0:
            return 'Cookie'
        elif row['Price_Per_Unit'] == 4.0:
            return 'Sandwich'
        elif row['Price_Per_Unit'] == 1.5:
            return 'Tea'
        else:
            return 'Juice'
    else:
        return row['Item']

df['Item'] = df.apply(right_item, axis=1)

df.groupby(['Item', 'Price_Per_Unit'])['Transaction_ID'].count()

Item      Price_Per_Unit
Cake      3.0               1254
Coffee    2.0               1185
Cookie    1.0               1088
Juice     3.0               1065
Salad     5.0               1150
Sandwich  4.0               1229
Smoothie  4.0                990
Tea       1.5               1073
Name: Transaction_ID, dtype: int64

In [15]:
def right_Quantity(row):
    if pd.isna(row['Total_Spent']) or row['Total_Spent'] == 0 or row['Total_Spent'] == 'nan':
        if not pd.isna(row['Quantity']) and row['Price_Per_Unit'] != 0:
            return row['Quantity'] * row['Price_Per_Unit']
        else: 
            return 0
    else:
        return row['Total_Spent']

df['Total_Spent'] = df.apply(right_Quantity, axis=1)

df.info()

<class 'pandas.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    9540 non-null   str           
 1   Item              9540 non-null   str           
 2   Quantity          9086 non-null   float64       
 3   Price_Per_Unit    9034 non-null   float64       
 4   Total_Spent       9521 non-null   float64       
 5   Payment_Method    9540 non-null   str           
 6   Location          9540 non-null   str           
 7   Transaction_Date  9540 non-null   datetime64[us]
 8   Season            9540 non-null   str           
dtypes: datetime64[us](1), float64(3), str(5)
memory usage: 745.3 KB


In [16]:
# after itms i will clean the price per unit

def right_price(row):
    if pd.isna(row['Price_Per_Unit']) or row['Price_Per_Unit'] == 0:
        if row['Item'] == 'Cake':
            return 3.0
        elif row['Item'] == 'Coffee':
            return 2.0
        elif row['Item'] == 'Salad':
            return 5.0
        elif row['Item'] == 'Cookie':
            return 1.0
        elif row['Item'] == 'Sandwich':
            return 4.0
        elif row['Item'] == 'Tea':
            return 1.5
        else:
            return 0
    else:
        return row['Price_Per_Unit']


df['Price_Per_Unit'] = df.apply(right_price, axis=1)

df.groupby(['Price_Per_Unit', 'Item'])['Transaction_ID'].count()

Price_Per_Unit  Item    
0.0             Juice        110
                Smoothie      58
1.0             Cookie      1147
1.5             Tea         1135
2.0             Coffee      1239
3.0             Cake        1305
                Juice       1065
4.0             Sandwich    1277
                Smoothie     990
5.0             Salad       1214
Name: Transaction_ID, dtype: int64

In [17]:
#Know I will change the price of each item with the most common price
medain = df['Price_Per_Unit'].median()

df['Price_Per_Unit'] = df['Price_Per_Unit'].replace(0, medain)

In [18]:
df['Price_Per_Unit'].value_counts()

Price_Per_Unit
3.0    2538
4.0    2267
2.0    1239
5.0    1214
1.0    1147
1.5    1135
Name: count, dtype: int64

In [19]:
mask = df['Total_Spent'].isna() & df['Quantity'].notna() & df['Price_Per_Unit'].notna()
df.loc[mask, 'Total_Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price_Per_Unit']

mask = df['Price_Per_Unit'].isna() & df['Total_Spent'].notna() & df['Quantity'].notna()
df.loc[mask, 'Price_Per_Unit'] = df.loc[mask, 'Total_Spent'] / df.loc[mask, 'Quantity']

mask = df['Quantity'].isna() & df['Total_Spent'].notna() & df['Price_Per_Unit'].notna()
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total_Spent'] / df.loc[mask, 'Price_Per_Unit']

df.info()


<class 'pandas.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    9540 non-null   str           
 1   Item              9540 non-null   str           
 2   Quantity          9540 non-null   float64       
 3   Price_Per_Unit    9540 non-null   float64       
 4   Total_Spent       9540 non-null   float64       
 5   Payment_Method    9540 non-null   str           
 6   Location          9540 non-null   str           
 7   Transaction_Date  9540 non-null   datetime64[us]
 8   Season            9540 non-null   str           
dtypes: datetime64[us](1), float64(3), str(5)
memory usage: 745.3 KB


In [20]:
mean = df['Quantity'].mean()

df['Quantity'] = df['Quantity'].replace(0, mean)


#fixing the data type of quantity 

df['Quantity'] = df['Quantity'].astype(int)

df.info()

<class 'pandas.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    9540 non-null   str           
 1   Item              9540 non-null   str           
 2   Quantity          9540 non-null   int64         
 3   Price_Per_Unit    9540 non-null   float64       
 4   Total_Spent       9540 non-null   float64       
 5   Payment_Method    9540 non-null   str           
 6   Location          9540 non-null   str           
 7   Transaction_Date  9540 non-null   datetime64[us]
 8   Season            9540 non-null   str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 745.3 KB


In [21]:
df.to_csv('cleaned_cafe_sales.csv', index=False)